# Explore catalog + events + cross-media calendar

This notebook is the first pass for the RAG prediction project.

**Question it sets up:** which storefront games are likely to see attention around upcoming physical events, digital events, and entertainment releases — and **when to promote them** during the equivalent runtime (see `02_promotion_strategies.ipynb`).

Sources:

- `data/raw/game_products.csv` — storefront SKUs with release dates
- `data/raw/events_and_adaptations.ods` — seed calendar
- **Live database (daily):** Wikipedia product/event pages and Wikidata, covering **2026 through 2030**
- **Announced / unreleased products:** TBA and dated announcements are kept in the product set (not only shipped catalog SKUs)
- **Event correlations:** announced titles are linked into matching event windows and also create release-window events
- **Broad coverage registry:** regional/global physical events, digital showcases/sales, esports, conferences, game jams, and cross-media releases in theatrical, TV, streaming, animation, anime, audio, reality, and live-stage formats

Run `python -m src.database` or the daily brief job. It overwrites the processed event, adaptation, upcoming-game, announced-game, promotion, and live JSON datasets with refreshed metadata.

In [1]:
from collections import Counter
from datetime import date
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.database import live_meta, refresh_live_database
from src.documents import build_retrieval_corpus, keyword_retrieve, upcoming_products
from src.load_data import load_adaptations, load_catalog, load_events
from src.match import catalog_around_window, franchise_queries, match_catalog
from src.paths import DATA_PROCESSED
from src.promote import build_plans, write_promotion_csv

print("project root:", ROOT)
if not live_meta():
    print("building live database (Wikipedia product + event pages)…")
    print(refresh_live_database(fetch=True))
else:
    print("live database:", live_meta())

project root: /Users/driffle/RAGPredictionGameProject


## Load both datasets

The catalog loader drops gift cards and treats `2079` / `2080` placeholder dates as missing.

In [3]:
events = load_events()
adaptations = load_adaptations()
catalog = load_catalog(games_only=True, drop_placeholder_dates=True)

dated = [row for row in catalog if row["release_date_parsed"]]
print(f"events: {len(events)}")
print(f"adaptations: {len(adaptations)}")
print(f"game/dlc SKUs: {len(catalog)}")
print(f"game/dlc with a real release date: {len(dated)}")
print(f"unique canonical titles: {len({row['canonical_title'] for row in catalog})}")

events: 82
adaptations: 48
game/dlc SKUs: 55593
game/dlc with a real release date: 54358
unique canonical titles: 30747


## Catalog shape

Storefront mix, platforms, and how many dated games sit in 2026+ (the prediction window).

In [4]:
print("product_type")
for key, count in Counter(row["product_type"] for row in catalog).most_common():
    print(f"  {key:20} {count}")

print("\nplatform (top 10)")
for key, count in Counter(row["platform"] for row in catalog).most_common(10):
    print(f"  {key:20} {count}")

years = Counter(
    row["release_date_parsed"].year
    for row in dated
)
print("\nrelease year (dated games only)")
for year in sorted(years):
    if year >= 2020:
        print(f"  {year}  {years[year]}")

future = upcoming_products(catalog, after=date(2026, 8, 1))
print(f"\nreleases on/after 2026-08-01: {len(future)} SKUs")
print(f"unique titles in that window: {len({row['canonical_title'] for row in future})}")

product_type
  game                 43462
  dlc                  8850
  dlc,game             1923
  game point           1358

platform (top 10)
  Steam                40042
  Xbox Live            9963
  Nintendo             1040
  EA Play              929
  Ubisoft Connect      811
  PSN                  785
  GOG COM              576
  Epic Games           298
  Riot Games           224
  Other                198

release year (dated games only)
  2020  3472
  2021  3566
  2022  3517
  2023  5926
  2024  10462
  2025  4223
  2026  3119
  2027  16

releases on/after 2026-08-01: 702 SKUs
unique titles in that window: 274


## Events calendar

Two sections in the ODS: industry/sports/esports events, then film/TV/anime adaptations of game IPs.

In [5]:
print("event types")
for key, count in Counter(row["event_type"] for row in events).most_common():
    print(f"  {key:28} {count}")

print("\nstatus")
for key, count in Counter(row["status"] for row in events).most_common():
    print(f"  {key:28} {count}")

print("\nconfirmed / announced events")
for row in events:
    status = row["status"].lower()
    if "confirm" in status or "announce" in status:
        print(f"  {row['start_date']}  {row['event']:40}  {row['related_game']}")

event types
  Esports                      14
  Gaming Expo                  10
  Convention                   7
  Awards                       7
  Showcase                     6
  Digital Showcase             4
  Digital Festival             3
  Developer Conference         3
  Golf                         3
  Fan Convention               2
  Awards + Showcase            2
  Anime Convention             2
  Ice Hockey                   2
  Platform Showcase            2
  Publisher Showcase           2
  Publisher Event              1
  Digital Commerce             1
  Hardware Expo                1
  Festival                     1
  Basketball                   1
  Tennis                       1
  Football                     1
  Cycling                      1
  Fighting Games               1
  Gaming Convention            1
  Motorsport                   1
  American Football            1
  Baseball                     1

status
  Planning Window              71
  Confirmed         

In [6]:
print("adaptation mediums")
for key, count in Counter(row["medium"] for row in adaptations).most_common():
    print(f"  {key:28} {count}")

print("\ndated adaptations (not year-TBA windows)")
for row in adaptations:
    if row["start_date"] and row["start_date"] != row["end_date"]:
        continue
    if row["date_status"].lower().startswith("confirm") or "announced" in row["date_status"].lower():
        print(
            f"  {row['start_date']}  {row['ip_adaptation']:32}  {row['related_game']:22}  {row['distributor']}"
        )

adaptation mediums
  TV / OTT                     25
  Movie                        10
  Animated TV                  5
  Anime                        4
  Anime / OTT                  2
  Movie / Animation            1
  Anime / TV                   1

dated adaptations (not year-TBA windows)
  2026-05-12  Devil May Cry S2                  Devil May Cry           Netflix
  2026-06-05  Among Us                          Among Us                Paramount+
  2026-10-16  Street Fighter                    Street Fighter          Paramount
  2026-12-23  Angry Birds Movie 3               Angry Birds             Sony
  2027-03-19  Sonic the Hedgehog 4              Sonic                   Paramount
  2027-04-30  The Legend of Zelda               Zelda                   Sony Pictures / Nintendo
  2027-07-23  A Minecraft Movie sequel          Minecraft               Warner Bros.
  2027-11-10  Helldivers                        Helldivers              Sony Pictures
  2028-03-03  Elden Ring          

## Franchise → catalog matches

Generic related-game values (`Multi-platform`, `PC / Steam`, platform lists) are skipped. Named IPs are matched against catalog titles.

In [7]:
print(f"{'event/adaptation':40} {'query':22} {'unique titles':>14}")
print("-" * 80)
for row in events + adaptations:
    queries = franchise_queries(row.get("related_game"))
    if not queries:
        continue
    hits = match_catalog(catalog, queries, limit=500)
    label = row.get("event") or row.get("ip_adaptation")
    print(f"{label[:40]:40} {', '.join(queries)[:22]:22} {len(hits):14}")

event/adaptation                         query                   unique titles
--------------------------------------------------------------------------------
BlizzCon                                 warcraft, world of war             75
PUBG Global Championship                 pubg, battlegrounds                50
League of Legends Worlds                 league of legends                  64
CES Gaming                               nvidia, amd, asus, raz              3
Nintendo Direct                          mario, zelda, pokemon,             66
IEM Katowice                             counter-strike, counte              2
Six Invitational                         rainbow six, rainbow s             53
The Masters                              pga tour, pga 2k                   19
NBA Playoffs & Finals                    nba 2k                              1
Stanley Cup Playoffs                     nhl                                23
IIHF World Championship                  nhl      

In [ ]:
print("Resident Evil catalog titles (sample)")
for row in match_catalog(catalog, ["resident evil"], limit=12):
    print(f"  {row['release_date'] or 'undated':10}  {row['platform']:12}  {row['canonical_title']}")

print("\nStreet Fighter catalog titles (sample)")
for row in match_catalog(catalog, ["street fighter"], limit=12):
    print(f"  {row['release_date'] or 'undated':10}  {row['platform']:12}  {row['canonical_title']}")

## Releases near a confirmed event

Gamescom 2026 is a hard date. Catalog titles releasing within 21 days of that window are candidates for showcase / demand spikes.

In [ ]:
gamescom = next(row for row in events if row["event"] == "Gamescom" and row["start_date"].startswith("2026"))
near = catalog_around_window(
    catalog,
    gamescom["start_date_parsed"],
    gamescom["end_date_parsed"],
    pad_days=21,
)
print(
    f"Gamescom {gamescom['start_date']} → {gamescom['end_date']}: "
    f"{len(near)} unique titles in a ±21 day window\n"
)
for row in near[:25]:
    print(f"  {row['release_date']}  {row['platform']:12}  {row['canonical_title']}")

## RAG documents (keyword stand-in)

Each event, adaptation, upcoming product, and **timed promotion plan** becomes a retrieval chunk. Asking how to market EA Sports FC should return the football-tournament window and tactics — not a generic product card. Full walkthrough: `02_promotion_strategies.ipynb`.

In [ ]:
upcoming = upcoming_products(catalog, after=date(2026, 8, 1))
seen = set()
upcoming_unique = []
for row in upcoming:
    title = row["canonical_title"]
    if title in seen:
        continue
    seen.add(title)
    upcoming_unique.append(row)

plans = build_plans(events, adaptations, catalog)
documents = build_retrieval_corpus(events, adaptations, upcoming_unique, plans)
print(
    f"documents: {len(documents)} "
    f"({len(events)} events, {len(adaptations)} adaptations, "
    f"{len(upcoming_unique)} upcoming titles, {len(plans)} promotion plans)"
)

for query in [
    "How should we promote EA Sports FC during football tournaments?",
    "Capcom showcase Resident Evil Monster Hunter",
    "Nintendo Direct Zelda Mario Pokemon",
    "Street Fighter movie Paramount",
]:
    print(f"\nquery: {query}")
    for score, doc in keyword_retrieve(documents, query, limit=5):
        print(f"  {score:.2f}  [{doc['kind']:11}]  {doc['title']}")

## Export processed tables

Written to `data/processed/` for the later pipeline.

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


def write_csv(path, rows, fieldnames):
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            out = dict(row)
            out.pop("start_date_parsed", None)
            out.pop("end_date_parsed", None)
            out.pop("release_date_parsed", None)
            writer.writerow(out)
    print(f"wrote {path.name} ({len(rows)} rows)")


write_csv(
    DATA_PROCESSED / "events.csv",
    events,
    ["start_date", "end_date", "event", "category", "related_game", "event_type", "status"],
)
write_csv(
    DATA_PROCESSED / "adaptations.csv",
    adaptations,
    ["start_date", "end_date", "ip_adaptation", "medium", "distributor", "related_game", "date_status"],
)
write_csv(
    DATA_PROCESSED / "upcoming_games.csv",
    upcoming_unique,
    ["product_id", "canonical_title", "product_title", "product_sku", "product_type", "platform", "status", "release_date"],
)
promo_path = write_promotion_csv(plans)
print(f"wrote {promo_path.name}")

## Next

- Timed merchandising: `02_promotion_strategies.ipynb`
- Daily Google Trends + Wikipedia priorities: `03_daily_trend_priorities.ipynb` (`python3 -m src.daily_brief`)

Keep gift cards and `2080-01-01` placeholders out of training.